# Technical Analysis Evolution Lab — Colab Launcher

This notebook runs the research engine from the GitHub repository. It is designed for **historical research and paper-style evaluation**, not live trading.

In [ ]:
import os, sys, subprocess, yaml, json, pandas as pd
REPO='https://github.com/betaanoiar1-gif/Technical-Analysis-Evolution-Lab.git'
WORK='/content/Technical-Analysis-Evolution-Lab'
if not os.path.exists(WORK):
    subprocess.run(['git','clone','--depth','1',REPO,WORK], check=True)
%cd $WORK
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)


In [ ]:
# Research controls — change these values before running.
SYMBOL = 'BTC/USDT'
TIMEFRAME = '1h'
EXCHANGE = 'binance'
LIMIT = 3000
CONFIG_PATH = 'configs/default.yaml'
DATA_MODE = 'exchange'  # 'exchange' or 'upload'


In [ ]:
from taevo.data import fetch_exchange, load_csv
if DATA_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded))
    bundle = load_csv(csv_path, SYMBOL, TIMEFRAME)
else:
    bundle = fetch_exchange(SYMBOL, TIMEFRAME, LIMIT, EXCHANGE)
print(bundle.symbol, bundle.timeframe, bundle.source, len(bundle.frame), bundle.fingerprint[:16])
display(bundle.frame.tail())


In [ ]:
from taevo.experiment import run_experiment
from taevo.reporting import render_html
cfg = yaml.safe_load(open(CONFIG_PATH, encoding='utf-8'))
report = run_experiment(bundle, cfg, 'experiments/runs')
best = report['best_development'][0]
print('Best development candidate:', best['strategy'])
print('Validation score:', round(best['validation']['score'], 4))
print('Rejected:', best['rejected'], best['rejection_reasons'])
print('\nHoldout (never used for selection):')
print(json.dumps(best['holdout'], indent=2, default=str))
render_html(report, 'reports/latest.html')


In [ ]:
import matplotlib.pyplot as plt
from taevo.backtest import run_backtest
from taevo.schools import candidate_library
from taevo.strategies import Strategy
from taevo.data import split_three_way
train, validation, holdout = split_three_way(bundle.frame, cfg['validation']['train_ratio'], cfg['validation']['validation_ratio'])
selected_name = best['strategy']
candidate = next((c for c in candidate_library() if c.name == selected_name.split('+')[0]), None)
if candidate is not None:
    strategy = Strategy(candidate, tuple(selected_name.split('+')[1:]))
    result = run_backtest(holdout, strategy.signal(holdout), cfg['capital']['initial_usd'], cfg['costs']['fee_bps'], cfg['costs']['slippage_bps'])
    ax = result.equity.plot(figsize=(12,4), title='Selected candidate — HOLDOUT equity')
    ax.set_ylabel('Equity')
    plt.show()


In [ ]:
# Download the main research report to your Colab session.
from google.colab import files
files.download('reports/latest.html')
